## 04 - TabNet Model Training

### Import libraries and define paths

In [ ]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pytorch_tabnet.metrics import Metric
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef,
    precision_recall_fscore_support, roc_auc_score,
)
from sklearn.model_selection import train_test_split

### Step 1a - Configure paths, reproducibility and device

In [ ]:
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_FILE = PROJECT / 'data' / 'processed_data.npz'
MODELS_DIR = PROJECT / 'models'
MODEL_PREFIX = MODELS_DIR / 'tabnet_model'
MODEL_FILE = MODELS_DIR / 'tabnet_model.zip'
RESULTS_DIR = PROJECT / 'results' / 'tabnet'
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
device_name = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training device:', device_name)

### load the shared processed data and label encoder

In [ ]:
if not PROCESSED_FILE.exists():
    raise FileNotFoundError('Run Notebook 01 before training the model.')
processed_data = np.load(PROCESSED_FILE)
required = {'X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test'}
missing = required.difference(processed_data.files)
if missing:
    raise ValueError(f'Missing arrays in processed_data.npz: {sorted(missing)}')
X_train = processed_data['X_train'].astype(np.float32)
y_train = processed_data['y_train'].astype(np.int64)
X_val = processed_data['X_val'].astype(np.float32)
y_val = processed_data['y_val'].astype(np.int64)
X_test = processed_data['X_test'].astype(np.float32)
y_test = processed_data['y_test'].astype(np.int64)
print('X_train:', X_train.shape, 'X_val:', X_val.shape)
print('The test arrays are loaded but are reserved for the final evaluation cell.')

### Load labels, feature names and class distributions

In [ ]:
label_encoder = joblib.load(MODELS_DIR / 'label_encoder.joblib')
class_names = list(label_encoder.classes_)
number_of_classes = len(class_names)
BENIGN_INDEX = class_names.index('BENIGN')
all_labels = np.arange(number_of_classes)
feature_names = list(processed_data['feature_names']) if 'feature_names' in processed_data.files else [f'feature_{i}' for i in range(X_train.shape[1])]
print('Classes:', class_names)
distribution = pd.DataFrame({
    'train': np.bincount(y_train, minlength=number_of_classes),
    'validation': np.bincount(y_val, minlength=number_of_classes),
}, index=class_names)
display(distribution)

### tune tabNet using validation' macro-F1

In [ ]:
class ValidationMacroF1(Metric):
    _name = 'macro_f1'
    _maximize = True

    def __call__(self, y_true, y_score):
        return f1_score(y_true, np.argmax(y_score, axis=1), average='macro', zero_division=0)

BASE_CONFIG = {
    'n_d': 16, 'n_a': 16, 'n_steps': 5, 'gamma': 1.5,
    'lambda_sparse': 1e-4, 'learning_rate': 0.02,
    'batch_size': 4096, 'virtual_batch_size': 512,
    'max_epochs': 50, 'patience': 8, 'seed': RANDOM_STATE,
}
TUNING_GRID = [
    {'n_d': 16, 'n_a': 16, 'n_steps': 3, 'learning_rate': 0.02},
    {'n_d': 24, 'n_a': 24, 'n_steps': 4, 'learning_rate': 0.02},
    {'n_d': 32, 'n_a': 32, 'n_steps': 5, 'learning_rate': 0.01},
]

def build_tabnet(config):
    return TabNetClassifier(
        n_d=config['n_d'], n_a=config['n_a'], n_steps=config['n_steps'],
        gamma=config['gamma'], lambda_sparse=config['lambda_sparse'],
        optimizer_fn=torch.optim.Adam,
        optimizer_params={'lr': config['learning_rate']},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        scheduler_params={'step_size': 20, 'gamma': 0.9},
        mask_type='entmax', seed=config['seed'], verbose=0,
        device_name=device_name,
    )

def macro_f1_history(model):
    matching_keys = [key for key in model.history.keys() if 'macro_f1' in key]
    if not matching_keys:
        raise KeyError(f'Could not find macro-F1 in TabNet history: {list(model.history.keys())}')
    return model.history[matching_keys[0]]

### run the validation only hyperparameter search

In [ ]:
tuning_results = []
for trial_number, settings in enumerate(TUNING_GRID, start=1):
    trial_config = {**BASE_CONFIG, **settings, 'seed': RANDOM_STATE + trial_number}
    print(f'\nTrial {trial_number}/{len(TUNING_GRID)}: {settings}')
    set_seed(trial_config['seed'])
    trial_model = build_tabnet(trial_config)
    trial_start = time.perf_counter()
    trial_model.fit(
        X_train, y_train, eval_set=[(X_val, y_val)], eval_name=['validation'],
        eval_metric=[ValidationMacroF1], max_epochs=trial_config['max_epochs'],
        patience=trial_config['patience'], batch_size=trial_config['batch_size'],
        virtual_batch_size=trial_config['virtual_batch_size'], weights=1,
        drop_last=False,
    )
    trial_macro_f1 = f1_score(y_val, trial_model.predict(X_val), average='macro', zero_division=0)
    trial_history = macro_f1_history(trial_model)
    tuning_results.append({
        **settings, 'trial': trial_number,
        'best_validation_macro_f1': float(max(trial_history)),
        'final_validation_macro_f1': float(trial_macro_f1),
        'epochs_run': len(trial_history),
        'training_seconds': time.perf_counter() - trial_start,
    })

tuning_table = pd.DataFrame(tuning_results).sort_values('best_validation_macro_f1', ascending=False)
tuning_table.to_csv(RESULTS_DIR / 'tuning_results.csv', index=False)
display(tuning_table)
best_settings = tuning_table.iloc[0].to_dict()
FINAL_CONFIG = {**BASE_CONFIG, **{key: best_settings[key] for key in TUNING_GRID[0]}}
print('Selected by validation macro-F1:', FINAL_CONFIG)

### Train the final model and save validation history

In [ ]:
set_seed(RANDOM_STATE)
tabnet_model = build_tabnet(FINAL_CONFIG)
training_start = time.perf_counter()
tabnet_model.fit(
    X_train, y_train, eval_set=[(X_val, y_val)], eval_name=['validation'],
    eval_metric=[ValidationMacroF1], max_epochs=FINAL_CONFIG['max_epochs'],
    patience=FINAL_CONFIG['patience'], batch_size=FINAL_CONFIG['batch_size'],
    virtual_batch_size=FINAL_CONFIG['virtual_batch_size'], weights=1, drop_last=False,
)
training_seconds = time.perf_counter() - training_start
final_macro_f1_history = macro_f1_history(tabnet_model)
best_validation_macro_f1 = float(max(final_macro_f1_history))
tabnet_model.save_model(str(MODEL_PREFIX))
history = pd.DataFrame(dict(tabnet_model.history))
history.to_csv(RESULTS_DIR / 'training_history.csv', index=False)
with open(RESULTS_DIR / 'config.json', 'w') as file:
    json.dump(FINAL_CONFIG, file, indent=2)
print(f'Best validation macro-F1: {best_validation_macro_f1:.4f}')
print(f'Model saved: {MODEL_FILE}')

### Plot training loss and validation macro-F1

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(history.index + 1, history['loss'], label='Train loss')
axes[0].set_title('TabNet training loss')
axes[1].plot(history.index + 1, final_macro_f1_history, color='tab:green', label='Validation macro-F1')
axes[1].axhline(best_validation_macro_f1, color='grey', linestyle='--', label='Best validation macro-F1')
axes[1].set_title('Validation macro-F1')
for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.3)
    axis.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'learning_curves.png', dpi=150)
plt.show()

### Measure train/validation generalisation and validation-only explainability

In [ ]:
train_predictions = tabnet_model.predict(X_train)
val_predictions = tabnet_model.predict(X_val)
generalisation = pd.DataFrame([
    {
        'split': split_name,
        'accuracy': accuracy_score(targets, predictions),
        'balanced_accuracy': balanced_accuracy_score(targets, predictions),
        'macro_f1': f1_score(targets, predictions, average='macro', zero_division=0),
        'weighted_f1': f1_score(targets, predictions, average='weighted', zero_division=0),
    }
    for split_name, targets, predictions in [('train', y_train, train_predictions), ('validation', y_val, val_predictions)]
]).set_index('split')
generalisation.to_csv(RESULTS_DIR / 'generalisation.csv')
display(generalisation.round(4))

### Generate validation only TabNet explainability outputs

In [ ]:
explain_size = min(5000, len(y_val))
explain_indices, _ = train_test_split(
    np.arange(len(y_val)), train_size=explain_size,
    random_state=RANDOM_STATE, stratify=y_val,
)
explain_start = time.perf_counter()
explain_masks, explain_probabilities = tabnet_model.explain(X_val[explain_indices])
explain_seconds = time.perf_counter() - explain_start
global_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': tabnet_model.feature_importances_,
}).sort_values('importance', ascending=False)
global_importance.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)
np.savez_compressed(
    RESULTS_DIR / 'explainability_masks.npz',
    indices=explain_indices,
    masks=explain_masks.astype(np.float32),
    probabilities=explain_probabilities.astype(np.float32),
)
display(global_importance.head(20))

### Plot global feature importance

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(data=global_importance.head(20), x='importance', y='feature', color='tab:blue')
plt.title('TabNet feature importance - validation explainability subset')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_importance.png', dpi=150)
plt.show()

### Perform the single final test evaluation

In [ ]:
if device_name == 'cuda':
    torch.cuda.synchronize()
inference_start = time.perf_counter()
test_probabilities = tabnet_model.predict_proba(X_test)
if device_name == 'cuda':
    torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_start
test_predictions = test_probabilities.argmax(axis=1)
print(f'Test inference: {inference_seconds:.2f}s for {len(X_test):,} rows')

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, test_predictions, labels=all_labels, zero_division=0,
)
per_class_metrics = pd.DataFrame({
    'class': class_names,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'support': support,
})
per_class_metrics.to_csv(RESULTS_DIR / 'per_class_metrics.csv', index=False)
multiclass_metrics = {
    'accuracy': accuracy_score(y_test, test_predictions),
    'macro_precision': precision_recall_fscore_support(y_test, test_predictions, average='macro', zero_division=0)[0],
    'macro_recall': precision_recall_fscore_support(y_test, test_predictions, average='macro', zero_division=0)[1],
    'macro_f1': f1_score(y_test, test_predictions, average='macro', zero_division=0),
    'weighted_precision': precision_recall_fscore_support(y_test, test_predictions, average='weighted', zero_division=0)[0],
    'weighted_recall': precision_recall_fscore_support(y_test, test_predictions, average='weighted', zero_division=0)[1],
    'weighted_f1': f1_score(y_test, test_predictions, average='weighted', zero_division=0),
    'balanced_accuracy': balanced_accuracy_score(y_test, test_predictions),
}
display(pd.Series(multiclass_metrics).round(4))

### plot the final multiclass confusion matrices

In [ ]:
confusion = confusion_matrix(y_test, test_predictions, labels=all_labels)
row_normalised = confusion / np.maximum(confusion.sum(axis=1, keepdims=True), 1)
figure, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(confusion, cmap='Blues', annot=True, fmt='d', cbar=False, xticklabels=class_names, yticklabels=class_names, ax=axes[0])
sns.heatmap(row_normalised, cmap='Blues', annot=True, fmt='.2f', vmin=0, vmax=1, xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[0].set_title('TabNet confusion matrix - counts')
axes[1].set_title('TabNet confusion matrix - row-normalised recall')
for axis in axes:
    axis.set_xlabel('Predicted class')
    axis.set_ylabel('True class')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

### Save binary metrics, predictions, model size and final summary

In [ ]:
y_test_binary = (y_test != BENIGN_INDEX).astype(int)
test_predictions_binary = (test_predictions != BENIGN_INDEX).astype(int)
attack_score = 1.0 - test_probabilities[:, BENIGN_INDEX]
tn, fp, fn, tp = confusion_matrix(y_test_binary, test_predictions_binary, labels=[0, 1]).ravel()
binary_precision, binary_recall, binary_f1, _ = precision_recall_fscore_support(
    y_test_binary, test_predictions_binary, average='binary', zero_division=0,
)
binary_metrics = {
    'precision': binary_precision,
    'recall_detection_rate': binary_recall,
    'f1': binary_f1,
    'roc_auc': roc_auc_score(y_test_binary, attack_score),
    'pr_auc': average_precision_score(y_test_binary, attack_score),
    'false_positive_rate': fp / max(fp + tn, 1),
    'mcc': matthews_corrcoef(y_test_binary, test_predictions_binary),
    'true_negative': int(tn),
    'false_positive': int(fp),
    'false_negative': int(fn),
    'true_positive': int(tp),
}
display(pd.Series(binary_metrics).round(4))

### Save reusable metrics, predictions and model statistics

In [ ]:
trainable_parameters = sum(parameter.numel() for parameter in tabnet_model.network.parameters() if parameter.requires_grad)
efficiency = {
    'trainable_parameters': int(trainable_parameters),
    'model_file_mb': MODEL_FILE.stat().st_size / 1024 ** 2,
    'training_seconds': training_seconds,
    'inference_seconds': inference_seconds,
    'inference_ms_per_1000_rows': 1000 * 1000 * inference_seconds / len(X_test),
    'explainability_seconds': explain_seconds,
    'epochs_run': len(history),
    'best_validation_macro_f1': best_validation_macro_f1,
    'device': device_name,
}
metrics = {
    'model': 'TabNet',
    'selection_metric': 'validation_macro_f1',
    'class_weighting': 'pytorch_tabnet balanced training sample weights (weights=1)',
    'config': FINAL_CONFIG,
    'multiclass': {key: float(value) for key, value in multiclass_metrics.items()},
    'binary_benign_vs_attack': {
        key: (float(value) if isinstance(value, (float, np.floating)) else value)
        for key, value in binary_metrics.items()
    },
    'efficiency': efficiency,
}
with open(RESULTS_DIR / 'metrics.json', 'w') as file:
    json.dump(metrics, file, indent=2)
np.savez_compressed(
    RESULTS_DIR / 'test_predictions.npz',
    y_true=y_test,
    y_pred=test_predictions,
    probabilities=test_probabilities.astype(np.float32),
    class_names=np.array(class_names),
)
pd.DataFrame({'y_true': y_test, 'y_pred': test_predictions}).to_csv(
    RESULTS_DIR / 'test_predictions.csv', index=False,
)
summary = pd.concat([
    pd.Series(multiclass_metrics).add_prefix('multiclass_'),
    pd.Series(binary_metrics).add_prefix('binary_'),
    pd.Series(efficiency),
])
summary.to_csv(RESULTS_DIR / 'summary.csv', header=['value'])
print('Saved TabNet model, metrics, predictions and figures in', RESULTS_DIR)